# 도구 순서 설계 ① 소프트 규칙 — 입장권·탈출구·넛지 (GPT 버전)

클로드코드(CC)에는 도구 순서를 **"지정"하는 코드가 없다**. 강제력이 다른 2층 장치로 순서가 "설계"되어 있다:

| 층 | 강제력 | 위반하면 | 이 시리즈 |
|---|---|---|---|
| **소프트 규칙** (유도 설계) | 안 따라도 동작함 | 비효율일 뿐 | **이 노트북** |
| **하드 규칙** (코드 게이트) | 시도하면 코드가 거부 | 에러 → 모델이 복구 | ② `cc_tool_sequence_hard_rules.ipynb` |

이 노트북이 다루는 소프트 장치 전체 지도:

```
소프트 A (입장권 설계) — 구조로 유도
├─ 입장권 경사   : 필수 파라미터의 조달 난이도가 계단 (공짜 패턴 → 조달 경로 → 정확한 원문)
├─ 출력 부피 경사 : glob ≤100개 → grep ≤250줄 → read ≤2000줄
└─ 니치 선언     : 각 description이 "언제 나를 쓰는지"를 말함

소프트 B (문구 장치) — 말로 유도
├─ 탈출구       : "열린 탐색이면 agent_search로" (항상 서 있는 표지판)
├─ 결과 넛지    : 잘림·오타·재읽기·다중매칭 순간에 "다음에 뭘 하면 되는지"를 결과에 심음
└─ 방치 리마인더 : 도구 방치가 감지된 그 턴에만 system-reminder 주입
```

GPT(Responses API) 이식 포인트: 에러·넛지는 전부 `function_call_output` 문자열로 (— "에러 메시지는 로그가 아니라 프롬프트다"), tools 배열은 이름순 정렬 후 세션 내 동결 (OpenAI 캐시는 exact prefix match — tools 변경 = 전체 미스).

> 참고: `도구호출-순서설계-하드소프트.md` + `md_group/`(도구-연계-11종, 도구지침-분산기준-라우터, 도구-정렬-캐시보존, openai-toolsearch-kv-cache 등)

In [1]:
import cc_tools as cts
from cc_tools import FS, reset_fs, SoftSession as Session, SOFT_TOOLS as TOOLS, DEFERRED_TOOLS

# 소프트/하드 도구·세션은 한 파일 cc_tools.py에 함께 있다 (두 노트북이 같은 파일 사용).
# read/edit/write는 그 안에 한 벌뿐이고, 소프트는 nudges 스위치, 하드는 state 게이트 스위치만 다르다.
MODEL = cts.MODEL  # 기본 gpt-5-nano — 문구 유도 데모는 Session(model=...)으로 오버라이드
print("기본 MODEL:", MODEL)

기본 MODEL: gpt-5-nano


## 1. 목 파일시스템

재현성을 위해 인메모리 목 파일시스템을 쓴다. **이 노트북의 edit에는 하드 게이트(읽기 강제)가 일부러 없다** — 그런데도 4도구 사슬이 나오는지가 관전 포인트다.

In [2]:
# 공통 목 코드베이스(orderhub — FastAPI 주문 관리 백엔드, 40파일)로 FS 초기화.
# ★ 소프트 경로의 edit에는 하드 게이트(읽기 강제)가 일부러 없다(state 미전달) —
#   그런데도 4도구 사슬이 저절로 나오는지가 관전 포인트다.
print(f"seeded {reset_fs()} files")

seeded 40 files


## 2. 도구 구현 — 넛지가 심어지는 자리

CC는 규칙 하나를 4축(범위·시점·강제성·토큰)으로 분류해 4개 층에 배치한다 (`도구지침-분산기준-라우터.md`). 소프트 규칙이 앉는 자리는 그중 3곳:

| 조건 | 목적지 | 이 노트북의 예 |
|---|---|---|
| 전역 + 항상 | 시스템 프롬프트 | 병렬 호출 처방 |
| 단일 도구 + 그때만 | 도구 description | 니치 선언 · 탈출구 표지판 |
| 런타임 동적 | 결과·에러 메시지 | 잘림 넛지 · "혹시 X 파일을 찾으시나요?" · 재읽기 스텁 · 다중매칭 안내 |

원리 한 줄: **"권장은 말로, 필수는 코드로."** (필수=코드는 ② 하드 노트북에서)

아래 넛지·에러 문구는 CC 원문을 한국어로 옮긴 것이고, 상한 수치는 CC 값 그대로다.

In [3]:
# read_file·edit_file은 cc_tools.py의 '한 벌'을 스위치로 바인딩한 것:
# 소프트는 nudges=True(오타·재읽기 스텁·잘림 넛지) + state 없음(읽기 강제 게이트 없음).
# glob/grep/agent_search/todo_write는 소프트 전용. 넛지 문구·상한(100/250/2000)은 공통 상수.
from cc_tools import (glob_files, grep_files, soft_read_file as read_file,
                              soft_edit_file as edit_file, agent_search, todo_write,
                              SOFT_TOOL_IMPLS as TOOL_IMPLS)
print("상한:", cts.GLOB_LIMIT, cts.GREP_HEAD_LIMIT, cts.READ_MAX_LINES)
print("도구:", list(TOOL_IMPLS))

상한: 100 250 2000
도구: ['glob_files', 'grep_files', 'read_file', 'edit_file', 'agent_search', 'todo_write']


## 3. 도구 스키마 — 입장권 계단과 탈출구 표지판

**입장권** = 각 도구의 필수 파라미터. 입장권 가격이 계단이라 4도구 사슬이 저절로 나온다:

```
glob_files  입장권: pattern 하나          (모델이 지어낼 수 있음 = 공짜)   ┐
grep_files  입장권: pattern 하나          (공짜 — path/glob 전부 optional) ├→ 경로 확보 (Y자 합류)
read_file   입장권: ABSOLUTE file_path    (지어낼 수 없음 → 위에서 조달)  ←┘
edit_file   입장권: file_path + old_string(파일 원문과 정확 일치 → read 출력에서 조달)
```

- **니치 선언**: "이름 패턴으로 파일을" / "내용(CONTENT)을 검색" / "절대경로(ABSOLUTE)여야" — 각자 자기 자리를 말한다.
- **탈출구**: glob/grep description 끝의 CC 원문 번역 — *"여러 라운드의 glob과 grep이 필요할 수 있는 열린 탐색이라면 이 도구 대신 agent_search 도구를 사용하세요."* 시퀀스 규칙이 아니라 **경계 안내**(이 경로를 벗어나야 할 조건)다.
- **GPT 특화**: 변이 도구(edit)만 `strict: True`(서버 스키마 강제), tools는 이름순 정렬 후 동결(캐시).

In [4]:
# 동결 tools 6개 + 레지스트리(디퍼드) 2개. 레지스트리 도구(agent_search·todo_write)는 tools에
# 싣지 않는다: 스키마는 tool_search 결과(대화)로, 실행은 tool_invoke 게이트웨이로만
# (gpt_toolsearch_kv_cache.ipynb 디스패처와 동일 = 세션 내내 tools 불변 = 프리픽스 캐시 미스 0).
print("동결 tools:", [t["name"] for t in TOOLS])
print("디퍼드:", [t["name"] for t in DEFERRED_TOOLS])

동결 tools: ['edit_file', 'glob_files', 'grep_files', 'read_file', 'tool_invoke', 'tool_search']
디퍼드: ['agent_search', 'todo_write']


## 4. 에이전트 루프 + 방치 리마인더

- **시스템 프롬프트에 순서 지시는 0줄** — 병렬 호출 처방, `<system-reminder>` 채널 정의(CC `prompts.ts:190` 대응), **레지스트리 도구 고지**(agent_search·todo_write는 이름만 공개 — tool_search로 조회 후 tool_invoke로 실행), **열린 조사 위임 정책**(검색 도구 반복 대신 통째 위임 — nano용 상시 정책, 데모 4 관찰 참고)이 있다. 순서 지시는 여전히 0줄 — 순서는 오직 입장권 설계에서 나온다.
- **디스패처 게이트 (`gpt_toolsearch_kv_cache.ipynb`와 동일 설계)**: agent_search·todo_write는 tools 배열에 없고 클라이언트 레지스트리에만 있다 — 직접 emit 자체가 불가능하다. 스키마는 `tool_search` 결과(대화 내용)로 받고, 실행은 동결된 `tool_invoke` 게이트웨이로만 한다. tools 배열은 6개로 **세션 내내 불변** → 프리픽스 캐시 미스 0. 대가: 서버 strict 검증 불가 → 클라이언트 인자 검증 + 에러 리턴으로 자기교정 (CC ToolSearch·디퍼드 대응).
- **todo 모드 (데모 5 전용)**: `Session(todo_mode=True)`일 때만 **태스크 관리 권장**(CC 시스템 프롬프트의 주 동력 대응)과 방치 리마인더(보조 넛지, 목록 echo)가 켜진다. 데모 2~4은 순서 관찰에 집중하도록 꺼 둔다.
- **방치 리마인더**(소프트 B의 셋째 갈래): CC는 태스크 도구가 10턴+ 미사용이면 그 턴에 `<system-reminder>`를 주입한다. 주기 보고가 아니라 **방치 감지형** — 사건이 난 그 순간에만 꽂히는 동적 넛지다. 여기서는 데모 규모에 맞춰 임계값을 2라운드로 줄였다. 문구도 CC 원문의 한국어 축약 — "고려하세요 / 관련될 때만 / 해당 없으면 무시하세요" 같은 표현이 **스스로 소프트 규칙임을 명시**하는 게 특징.

In [5]:
# 시스템 프롬프트에 순서 지시는 0줄 — 병렬 처방 + <system-reminder> 채널 정의(CC prompts.ts:190)
# + 레지스트리 고지(agent_search·todo_write는 이름만 공개) + 열린 조사 위임 정책만 있다.
# 순서는 오직 입장권 설계에서 나온다. 방치 리마인더·태스크 관리 권장은 todo 모드(데모 5)에서만 켜진다.
from cc_tools import SOFT_SYSTEM_PROMPT
print(SOFT_SYSTEM_PROMPT)

당신은 사용자의 프로젝트에서 일하는 코딩 에이전트입니다. 프로젝트 파일은 /project 아래에 있으며 모든 경로는 절대경로입니다.

도구 결과와 사용자 메시지에는 <system-reminder> 태그나 다른 태그가 포함될 수 있습니다. 태그 안의 정보는 사용자가 아니라 시스템이 주입한 것이며, 태그가 등장한 도구 결과나 사용자 메시지와 직접적인 관련이 없을 수 있습니다.

다음 도구들은 레지스트리에만 있어 직접 호출할 수 없습니다: agent_search, todo_write. 필요하면 tool_search로 스키마를 조회한 뒤, tool_invoke(name=도구이름, arguments=스키마에 맞는 인자 객체)로 실행하세요.

프로젝트 전반을 훑어야 하는 열린 조사는 검색 도구를 여러 번 반복하지 말고, 조사를 통째로 맡길 수 있는 도구가 있으면 그쪽에 위임하세요.

한 응답에서 여러 도구를 호출할 수 있습니다. 호출들 사이에 의존성이 없다면 독립적인 도구 호출을 모두 병렬로 하세요. 단, 어떤 호출이 앞선 호출의 결과값에 의존한다면 병렬로 호출하지 말고 순차적으로 호출하세요.

한국어로 답하세요.


## 데모 1 — glob의 니치: "이름 패턴이 곧 답"일 때 무엇을 고르나

질문에 "파일명을 찾아라 / glob을 써라" 같은 도구·순서 힌트는 **한 줄도 없다**. 순수한 자연어 질문 하나 — *"테스트가 서비스별로 빠짐없이 있나?"* — 만 던진다. 관전 포인트는 이 질문의 **답이 파일 내용이 아니라 파일 목록의 형태**라는 것이다:

- 이건 특정 문자열을 **내용에서 찾는**(grep) 문제가 아니다. `tests/test_*.py`에 **어떤 파일이 있는지**를 `services/*.py` 목록과 **대조**하면 답이 나온다 — `payment_service`·`product_service`에는 테스트가 없다는 사실이 파일 이름만으로 드러난다.
- 그래서 glob의 니치 선언 — *"이름 패턴으로 파일을 찾아야 할 때 이 도구를 사용하세요"* — 가 정확히 이 질문을 가리킨다. 내용 검색(grep)으로 접근하면 오히려 먼 길이다.

데모 2가 "입장권 경사가 **순서**를 만든다"를 보여준다면, 데모 1은 그 앞단 — **니치 선언이 도구 선택 자체를 가른다**(내용이냐 이름이냐) — 를 보여준다. 여기에도 순서 지시는 0줄이고, "파일 찾아줘"류 직접 지시도 0줄이다.

> **gpt-5-nano 실측 관찰**: nano는 첫수로 `glob_files("/project/tests/*.py")`(이어서 `services/*.py`)를 던져 목록을 확보한 뒤, 서비스 대비 빠진 테스트를 짚는다. 목록을 잡고 나면 일부 파일은 `grep`/`read`로 내용까지 확인하러 가기도 하지만 **진입점은 내용검색이 아니라 이름 패턴 매칭(glob)**이다 — 니치 선언이 첫수를 갈랐다는 증거다. 반대로 같은 파일시스템에서 "어떤 코드가 예외를 던지나"처럼 **내용이 답인** 질문을 주면 진입점이 grep으로 바뀐다(데모 4에서 관찰).

In [6]:
s0 = Session()
_ = s0.ask("테스트가 어느 정도 갖춰져 있는지 확인해줘")

💬 테스트가 어느 정도 갖춰져 있는지 확인해줘

  🔧 glob_files({"pattern": "/project/**/test_*.py"})
     → /project/tests/test_utils.py …
  🔧 glob_files({"pattern": "/project/**/tests/*.py"})
     → /project/tests/test_utils.py …
  🔧 glob_files({"pattern": "/project/**/pytest.ini"})
     → 일치하는 파일이 없습니다
  🔧 glob_files({"pattern": "/project/**/pyproject.toml"})
     → /project/pyproject.toml
  🔧 glob_files({"pattern": "/project/**/requirements*.txt"})
     → 일치하는 파일이 없습니다
  🔧 glob_files({"pattern": "/project/.github/workflows/*"})
     → 일치하는 파일이 없습니다
  🔧 read_file({"file_path": "/project/tests/test_utils.py"})
     →      1	import pytest …
  🔧 read_file({"file_path": "/project/tests/test_order_service.py"})
     →      1	import pytest …
  🔧 read_file({"file_path": "/project/tests/test_auth_service.py"})
     →      1	from app.services.auth_service import _hash_password, authenticate …
  🔧 read_file({"file_path": "/project/tests/conftest.py"})
     →      1	import pytest …
  🔧 read_file({"file_path": "/pro

## 데모 2 — 입장권 설계: 4도구 사슬이 저절로 나온다

질문에 "먼저 검색하고 읽어라" 같은 순서 힌트는 전혀 없다. 하지만:

- read의 입장권(절대경로)은 지어낼 수 없다 → 공짜 입장권 도구(glob/grep)부터
- edit의 입장권(old_string = 파일 원문과 정확 일치)은 read 출력에서만 조달된다 → read가 edit보다 먼저

**glob/grep(경로 확보) → read(원문 확보) → edit(수정)** — 데이터 의존성만으로 이 순서가 유일해진다.

> **gpt-5-nano 실측 관찰**: nano는 이 데모에서 종종 read 없이 `edit(old_string="def authenticate(")`를 먼저 던진다 — old_string을 파일에서 조달하지 않고 **요청 문구에서 추측(입장권 위조)**한 것이다. 파이썬 `def 이름(` 문법이 뻔해서 위조가 가능했고, 이 노트북에는 하드 게이트가 없으므로 추측이 우연히 맞으면 그대로 통과한다. 소프트 규칙의 강제력이 모델의 신중함에 비례한다는 실증이자, ② 하드 노트북이 필요한 이유다. (gpt-5.5는 같은 조건에서 read를 먼저 했다)

In [7]:
s1 = Session()
_ = s1.ask("authenticate 라는걸 찾아서 이름을 verify_password로 바꿔줘.")

💬 authenticate 라는걸 찾아서 이름을 verify_password로 바꿔줘.

  🔧 grep_files({"pattern": "\\batenticate\\b", "path": "/project", "output_mode": "files_with_matches"})
     → 일치하는 내용이 없습니다
  🔧 grep_files({"pattern": "authenticate", "path": "/project", "output_mode": "files_with_matches"})
     → 3개 파일에서 발견 …
  🔧 read_file({"file_path": "/project/src/app/routers/auth_router.py"})
     →      1	from fastapi import APIRouter, Depends, HTTPException …
  🔧 read_file({"file_path": "/project/src/app/services/auth_service.py"})
     →      1	import hashlib …
  🔧 read_file({"file_path": "/project/tests/test_auth_service.py"})
     →      1	from app.services.auth_service import _hash_password, authenticate …
  🔧 grep_files({"pattern": "authenticate\\(", "path": "/project", "output_mode": "files_with_matches"})
     → 3개 파일에서 발견 …
  🔧 edit_file({"file_path": "/project/src/app/services/auth_service.py", "old_string": "def authenticate(username:)
     → /project/src/app/services/auth_service.py 파일이 수정되었습니다. 1곳을

## 데모 3 — 결과 넛지 가족 (스크립트 검증)

실패·성공 결과에 "다음에 뭘 하면 되는가"를 심는 장치들. LLM 없이 직접 확인:

- ① **잘림 넛지**: glob 100개 초과 → "더 구체적인 경로나 패턴을 사용해 보세요."
- ② **재읽기 절약 스텁**: 안 바뀐 파일 재읽기 → 본문 대신 스텁 한 줄
- ③ **오타 넛지**: 없는 경로 → "혹시 X 파일을 찾으시나요?"
- ④ **에러 리다이렉트**: 2000줄 초과 파일 → "offset/limit으로 필요한 부분만 읽거나, 통독 대신 검색하세요"
- ⑤ **다중매칭 넛지**: 복구 방법 2가지(replace_all / 더 넓은 컨텍스트)를 정확히 지시

In [ ]:
import re as _re

print("① 잘림 넛지 — 테스트 파일 120개를 만들고 glob:")
for i in range(120):
    FS[f"/project/tests/test_{i:03}.py"] = {"content": f"def test_{i}():\n    assert True\n",
                                            "mtime": cts._tick()}
lines = glob_files("/project/tests/*.py").splitlines()
print(f"   반환 줄 수: {len(lines)}  (경로 100개 + 넛지 1줄)")
print(f"   마지막 줄: {lines[-1]}\n")

print("② 재읽기 절약 스텁 — 같은 파일을 두 번 읽으면:")
_ = read_file("/project/src/app/services/auth_service.py")
print("  ", read_file("/project/src/app/services/auth_service.py"), "\n")

print("③ 오타 경로 넛지:")
print("  ", read_file("/project/src/app/services/auth_servic.py"), "\n")

print("④ 2000줄 초과 파일 리다이렉트:")
FS["/project/logs/big.log"] = {"content": "\n".join(f"line {i}" for i in range(1, 2501)),
                               "mtime": cts._tick()}
out = read_file("/project/logs/big.log")
print("   첫 줄:", out.splitlines()[0].strip())
print("   끝 줄:", out.splitlines()[-1], "\n")

print("⑤ 다중매칭 넛지:")
FS["/project/tmp_dup.py"] = {"content": "print(x)\nprint(x)\n", "mtime": cts._tick()}
print("  ", edit_file("/project/tmp_dup.py", "print(x)", "print(y)"))

# 데모용 합성 파일만 정리 — 공통 tests/의 실제 테스트(test_auth_service 등)는 건드리지 않는다
for p in [p for p in list(FS)
          if _re.search(r"/tests/test_\d{3}\.py$", p) or p in ("/project/logs/big.log", "/project/tmp_dup.py")]:
    del FS[p]

## 데모 4 — 탈출구 표지판 + 디퍼드 로드 (모델 민감도 관찰)

glob/grep description 끝의 탈출구 표지판은 `agent_search`를 가리키는데, **agent_search는 tools 배열에 없다**(레지스트리 전용 — 이름만 고지됨). 위임을 택한 모델은 `tool_search("select:agent_search")`로 스키마를 조회한 뒤 `tool_invoke`로 실행한다 — **검색 → 실행 2단 순서가 구조적으로 강제**된다(스키마 없이는 직접 emit 자체가 불가능한, 하드에 가까운 게이트). 캐시 설계는 `gpt_toolsearch_kv_cache.ipynb`의 디스패처와 동일(tools 동결 = 미스 0). 서브에이전트는 내부에서 여러 라운드를 돌고 **요약만** 반환한다 (위임→요약).

> **모델별 실측 관찰 (이 데모의 핵심)**: 이 데모의 요점은 "위임이 나오는가"가 **모델의 문구 준수력에 달렸다**는 것이다. gpt-5.5는 description 끝 탈출구 표지판만으로 위임을 택했고, gpt-5.4-mini도 시스템 프롬프트의 위임 정책과 함께 위임했다. 반면 **이 셀에서 쓰는 `gpt-4o-mini`는 (gpt-5-nano도 마찬가지로) 표지판·정책이 모두 서 있는데도 위임하지 않고 수동 grep·read 스윕을 그라인딩한다** — 여러 롤에서 재현됐다(아래 셀 출력이 그 한 롤 — `tool_search`/`tool_invoke→agent_search`가 전혀 등장하지 않고, `grep_files`·`read_file`을 직접 여러 번 호출해 훑는다). 위임을 안 하면 롤에 따라 검색을 얕게 끝내 요약이 부정확해지기도 한다(파일명 모드만 보고, 실제로는 여러 파일에 있는 `raise`를 "없음"이라 결론내는 식). 최종 요약 답변 자체는 그럴듯하지만, 탈출구가 가리키는 위임 경로는 전혀 밟지 않는다. **소프트 규칙(문구 장치)의 강제력은 "안 따라도 동작"하기 때문에 정확히 모델의 순응력만큼만 나온다** — 이 데모는 그 상한을 gpt-4o-mini로 직접 보여준다. 위임이 실제로 나오는 모습을 보려면 이 셀의 `model=`을 상위 모델로 바꿔 실행하면 된다. (상시 위임 정책을 사용자 발화가 아니라 시스템 프롬프트에 두는 것도 원칙 — 4축 라우터: 전역 정책 → 시스템 프롬프트)

In [ ]:
s2 = Session(model="gpt-4o-mini")  # 문구 유도 데모 — 위임 비순응 실측 관찰용 (위 주석 참고)
_ = s2.ask("이 프로젝트 전반의 예외 처리 방식(try/except, raise 등)이 어떤지 조사해서 요약해줘. "
           "나는 결과 요약만 보면 돼.")

## 데모 5 — 방치 리마인더: 멀티스텝 작업 중 주입

이 데모의 세션만 `Session(todo_mode=True, model="gpt-5.4-mini")` — 태스크 관리 권장(CC의 주 동력)과 방치 리마인더가 켜진다 (문구 유도 데모라 상위 모델, 데모 4 관찰 주석 참고). todo_write 역시 레지스트리 전용이라 모델은 먼저 `tool_search`로 스키마를 조회하고 `tool_invoke`로 기록해야 한다. 기대 흐름: **조회 → 계획 기록 → 진행하며 상태 갱신**. 이후 상태 갱신 없이 2라운드가 지나면 보조 넛지인 방치 리마인더가 발동하고, 이때 **현재 todo 목록이 함께 echo**된다(CC와 동일 — 목록이 비어 있으면 넛지만). 모델이 리마인더를 받고 갱신을 재개하는지, 아니면 처음부터 부지런히 갱신해 리마인더가 아예 안 뜨는지 관찰 — 어느 쪽이든 "방치가 감지된 순간에만 꽂힌다"는 메커니즘이 요점이다.

> **왜 시스템 프롬프트 권장이 필요한가 (nano 실측)**: CC에서 todo 사용을 견인하는 주 동력은 시스템 프롬프트의 태스크 관리 섹션(상시·강한 권장)이고, 방치 리마인더는 보조 장치다. 보조 넛지 혼자로는 nano가 반응하지 않는 것을 실측으로 확인했다 — 리마인더 문구 자체가 "해당 없으면 무시하라"는 무시 허가를 주는 조건부이기 때문이기도 하다.

In [ ]:
s3 = Session(todo_mode=True, model="gpt-5.4-mini")  # 문구 유도 데모 — 상위 모델
_ = s3.ask("세 가지 작업을 순서대로 해줘: "
           "(1) config.py의 DEBUG를 False로 바꾸고, "
           "(2) common.py의 TODO 주석을 제거하고, "
           "(3) README.md 첫 문단의 설명 문장 끝에 ' (설정 점검: DEBUG off)'를 덧붙여줘.", max_rounds=22)

## 정리 — 소프트 규칙 배치표

| 장치 | 갈래 | 이 코드에서의 위치 | 데모 |
|---|---|---|---|
| 입장권 경사 (공짜 패턴 → 조달 경로 → 정확 원문) | 소프트 A | 각 스키마 `required` + edit의 old_string | 데모 2 |
| 출력 부피 경사 (100개 → 250줄 → 2000줄) | 소프트 A | `GLOB_LIMIT`/`GREP_HEAD_LIMIT`/`READ_MAX_LINES` | 데모 3-①④ |
| 니치 선언 | 소프트 A | description ("이름 패턴으로" / "내용을 검색") | 데모 1·2 |
| 기본값이 만드는 순서 | 소프트 A | grep `output_mode` 기본 files_with_matches | (구현 주석) |
| 탈출구 (표지판 — 항상 상주) | 소프트 B | glob/grep description 끝 문장 | 데모 4 |
| 결과 넛지 (사건 순간에만) | 소프트 B | 잘림·스텁·오타·리다이렉트·다중매칭 | 데모 3 |
| 디스패처 게이트 (검색→실행 2단 강제) | 하드+소프트 | `tool_search`(스키마 조회) + `tool_invoke`(실행 게이트웨이) | 데모 4·5 |
| 성공 넛지 | 소프트 B | todo_write 결과 문구 ("계속 todo 목록으로…") | 데모 5 |
| 방치 리마인더 (방치 감지형) | 소프트 B | Session의 `<system-reminder>` 주입 | 데모 5 |
| 병렬 처방 + SR 채널 정의 + 디퍼드 고지 | 시스템 프롬프트 | `BASE_SYSTEM_PROMPT` (순서 지시는 0줄) | 전체 |
| 태스크 관리 권장 (todo 주 동력) | todo 모드 | `TODO_SYSTEM_EXTRA` — `Session(todo_mode=True)` | 데모 5 |
| tools 동결 + 이름순 정렬 | GPT 특화 | exact prefix 캐시 보존 — 디스패처 덕에 세션 내 구조적 미스 0 | 전체 |
| strict 하이브리드 | GPT 특화 | 변이 도구만 `strict: True` | 전체 |

이 노트북에 **없는** 것 = 하드 규칙. edit에 "읽기 강제" 게이트가 없어서, 모델이 대화 기억만 믿고 낡은 old_string으로 edit해도 **막을 방법이 없다** (우연히 매칭되면 잘못된 수정이 통과). 순서를 어기면 사고가 나는 지점은 코드 게이트가 필요하다 → ② `cc_tool_sequence_hard_rules.ipynb`.

**참고 문서**: `도구호출-순서설계-하드소프트.md` · `md_group/도구-연계-11종.md` · `도구지침-분산기준-라우터.md` · `도구-정렬-캐시보존.md` · `도구결과-가공-시스템.md` · `openai-toolsearch-kv-cache.md`